# Real-world Data Wrangling

In this project, you will apply the skills you acquired in the course to gather and wrangle real-world data with two datasets of your choice.

You will retrieve and extract the data, assess the data programmatically and visually, accross elements of data quality and structure, and implement a cleaning strategy for the data. You will then store the updated data into your selected database/data store, combine the data, and answer a research question with the datasets.

Throughout the process, you are expected to:

1. Explain your decisions towards methods used for gathering, assessing, cleaning, storing, and answering the research question
2. Write code comments so your code is more readable

## 1. Gather data

In this section, you will extract data using two different data gathering methods and combine the data. Use at least two different types of data-gathering methods.

### **1.1.** Problem Statement


- In this project, I’ll analyze the relationship between health expenditure and diabetes prevalence by combining datasets on global health spending and diabetes health indicators. Specifically, I aim to explore if higher healthcare spending correlates with lower rates of diabetes and related health outcomes.

### **1.2.** Gather at least two datasets using two different data gathering methods

List of data gathering methods:

- Download data manually
- Programmatically downloading files
- Gather data by accessing APIs
- Gather and extract data from HTML files using BeautifulSoup
- Extract data from a SQL database

Each dataset must have at least two variables, and have greater than 500 data samples within each dataset.

For each dataset, briefly describe why you picked the dataset and the gathering method (2-3 full sentences), including the names and significance of the variables in the dataset. Show your work (e.g., if using an API to download the data, please include a snippet of your code). 

Load the dataset programmtically into this notebook.

#### **Dataset 1**
Dataset 1: CDC Diabetes Health Indicators

Type: CSV file from the UCI Machine Learning Repository.
Method: Manual download.
Variables:
Diabetes_binary: 0 = No diabetes, 1 = Pre-diabetes or diabetes
BMI: Body Mass Index

In [ ]:
diabetes_data = pd.read_csv('path_to_diabetes_data.csv')
print(diabetes_data.head())

#### Dataset 2

Dataset 2: Health Expenditure World Bank 
Type: JSON (accessed via World Bank API)

Method: Programmatic download from World Bank API

Variables:

Country: Country name
Health Expenditure % GDP: Healthcare expenditure as a percentage of GDP
Year: Year of data observation

In [1]:
import requests

# Define API endpoint to fetch health expenditure data from World Bank
url = 'http://api.worldbank.org/v2/country/all/indicator/SH.XPD.CHEX.GD.ZS?format=json&per_page=1000&date=2010:2020'

# Request data from API
response = requests.get(url)
if response.status_code == 200:
    data = response.json()

    # Parse JSON data into a list of dictionaries for DataFrame conversion
    records = [
        {
            "Country": entry['country']['value'],
            "Country Code": entry['countryiso3code'],
            "Year": entry['date'],
            "Health Expenditure % GDP": entry['value']
        }
        for entry in data[1] if entry['value'] is not None
    ]
    health_exp_data = pd.DataFrame(records)
    print(health_exp_data.head())
    
    # Save to CSV for further use
    health_exp_data.to_csv("health_expenditure_api_data.csv", index=False)
else:
    print("Failed to fetch data:", response.status_code)


                       Country Country Code  Year  Health Expenditure % GDP
0  Africa Eastern and Southern          AFE  2020                  5.881779
1  Africa Eastern and Southern          AFE  2019                  5.830223
2  Africa Eastern and Southern          AFE  2018                  5.809334
3  Africa Eastern and Southern          AFE  2017                  5.953892
4  Africa Eastern and Southern          AFE  2016                  6.097986


Optional data storing step: You may save your raw dataset files to the local data store before moving to the next step.

In [ ]:
#Optional: store the raw data in your local data store

## 2. Assess data

Assess the data according to data quality and tidiness metrics using the report below.

List **two** data quality issues and **two** tidiness issues. Assess each data issue visually **and** programmatically, then briefly describe the issue you find.  **Make sure you include justifications for the methods you use for the assessment.**

### Quality Issue 1: Missing Values in BMI Column

In [ ]:
# Check for missing values visually
diabetes_data.info()  # View count of non-null values in each column


In [ ]:
# Calculate percentage of missing values in 'BMI' column
missing_bmi = diabetes_data['BMI'].isnull().sum() / len(diabetes_data) * 100
print(f"Missing 'BMI' values: {missing_bmi:.2f}%")


- The BMI column has missing values, which can affect the accuracy of our analysis. Missing BMI data may bias our results since it represents an important predictor for diabetes, and excluding these rows could remove key observations. Therefore, it’s necessary to handle or impute these values to maintain data quality.

### Quality Issue 2: Inconsistent Country Codes in health_exp_data

In [ ]:
# Display unique country codes and names in health expenditure data
print(health_exp_data['Country Code'].unique())


In [ ]:
# Check for consistency in country codes using regular expressions or predefined list
valid_codes = health_exp_data['Country Code'].str.match(r'^[A-Z]{3}$')
inconsistent_codes = health_exp_data[~valid_codes]
print("Inconsistent Country Codes:")
print(inconsistent_codes)


- Issue and justification: Country codes in health_exp_data have inconsistencies, which can cause issues during merging with diabetes_data that relies on standardized country names or codes. Ensuring that country codes match a three-letter ISO format is essential for accurate joining and comparison.

### Tidiness Issue 1: Multiple Variables in One Column (Health Expenditure % GDP and Year)

In [ ]:
# Sample of health expenditure data to check for year and percentage format
print(health_exp_data[['Year', 'Health Expenditure % GDP']].head(10))


In [ ]:
# Checking data types to confirm columns are separated but linked as needed
print(health_exp_data.dtypes)


- Issue and justification: The Health Expenditure % GDP values are associated with a specific Year, making it critical to use both columns together. Tidying this data might involve unstacking year-specific data into separate columns or clarifying the time series format for analysis.

### Tidiness Issue 2: Multiple Mismatched Column Names for Country 

In [ ]:
# Print column names of both datasets
print("Diabetes Data Columns:", diabetes_data.columns)
print("Health Expenditure Data Columns:", health_exp_data.columns)



In [ ]:
# Check if column names are consistent for merging
diabetes_countries = set(diabetes_data['Country'])
health_countries = set(health_exp_data['Country'])
inconsistent_names = diabetes_countries.symmetric_difference(health_countries)
print("Inconsistent Country Names:", inconsistent_names)


- Issue and justification: Inconsistencies in Country column naming conventions or formats can prevent merging and analysis. Standardizing column names or renaming columns as necessary is crucial to maintain a tidy dataset for efficient data joining and further analysis.

## 3. Clean data
Clean the data to solve the 4 issues corresponding to data quality and tidiness found in the assessing step. **Make sure you include justifications for your cleaning decisions.**

After the cleaning for each issue, please use **either** the visually or programatical method to validate the cleaning was succesful.

At this stage, you are also expected to remove variables that are unnecessary for your analysis and combine your datasets. Depending on your datasets, you may choose to perform variable combination and elimination before or after the cleaning stage. Your dataset must have **at least** 4 variables after combining the data.

In [ ]:
diabetes_data_copy = diabetes_data.copy()
health_exp_data_copy = health_exp_data.copy()

### **Quality Issue 1: Missing Values in BMI Column**

In [ ]:
# Filling missing values in BMI with the mean BMI for simplicity
diabetes_data['BMI'].fillna(diabetes_data['BMI'].mean(), inplace=True)


In [ ]:
# Checking if there are any remaining missing values in BMI
print("Missing values in BMI column after cleaning:", diabetes_data['BMI'].isnull().sum())


- Justification: Imputing missing values with the mean of the BMI column preserves the distribution of the data without losing records. This approach is suitable for maintaining dataset size and retaining relevant features for analysis.

### **Quality Issue 2: Inconsistent Country Codes in health_exp_data**

In [ ]:
# Standardizing all country codes to upper case, assuming three-letter ISO codes
health_exp_data['Country Code'] = health_exp_data['Country Code'].str.upper()


In [ ]:
# Checking unique country codes to ensure they are consistent
print("Unique country codes after cleaning:", health_exp_data['Country Code'].unique())


- Justification: By ensuring that all Country Code values are standardized to uppercase, we reduce the risk of mismatches during data merging, making the dataset more reliable and uniform for analysis.

### **Tidiness Issue 1: Multiple Variables in One Column (Health Expenditure % GDP and Year)**

In [ ]:
# Converting 'Year' to index for time series analysis, ensuring clear association with each GDP value
health_exp_data.set_index('Year', inplace=True)


In [ ]:
# Displaying a sample of the data to confirm successful reformatting
print(health_exp_data.head())


- Justification: Setting Year as the index clarifies each Health Expenditure % GDP entry, allowing easier temporal analysis without the need to repeatedly reference both columns.

### **Tidiness Issue 2: Mismatched Column Names for Country**

In [1]:
# Standardizing column name across datasets for a seamless merge
diabetes_data.rename(columns={'Country': 'Country Name'}, inplace=True)


In [2]:
# Confirming the renamed column matches in both datasets
print("Diabetes Data Columns:", diabetes_data.columns)
print("Health Expenditure Data Columns:", health_exp_data.columns)


- Justification: Ensuring consistent naming conventions for country columns across datasets simplifies merging operations and eliminates errors during analysis.

### **Remove unnecessary variables and combine datasets**

Strategy and Justification: Only keep essential columns that contribute directly to the research question. This step removes irrelevant data and focuses on crucial metrics.

In [ ]:
# Select relevant columns
diabetes_data = diabetes_data[['Country Name', 'Diabetes Prevalence', 'BMI']]
health_exp_data = health_exp_data[['Country Name', 'Health Expenditure % GDP']]

# Merging datasets on 'Country Name'
combined_data = pd.merge(diabetes_data, health_exp_data, on='Country Name')

# Display merged data sample to ensure accurate merging
print(combined_data.head())


## 4. Update your data store
Update your local database/data store with the cleaned data, following best practices for storing your cleaned data:

- Must maintain different instances / versions of data (raw and cleaned data)
- Must name the dataset files informatively
- Ensure both the raw and cleaned data is saved to your database/data store

In [ ]:
# Saving raw data
diabetes_data.to_csv('raw_diabetes_data.csv', index=False)
health_exp_data.to_csv('raw_health_expenditure_data.csv', index=False)

# Saving cleaned combined data
combined_data.to_csv('cleaned_combined_health_data.csv', index=False)


## 5. Answer the research question

### **5.1:** Define and answer the research question 
Going back to the problem statement in step 1, use the cleaned data to answer the question you raised. Produce **at least** two visualizations using the cleaned data and explain how they help you answer the question.

Question: Is there a relationship between diabetes prevalence and healthcare expenditure as a percentage of GDP across different countries?

Visual 1: Scatter Plot of Diabetes Prevalence vs. Health Expenditure % GDP

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Scatter plot
plt.figure(figsize=(10, 6))
sns.scatterplot(data=combined_data, x='Health Expenditure % GDP', y='Diabetes Prevalence')
plt.title('Diabetes Prevalence vs Health Expenditure % GDP')
plt.xlabel('Health Expenditure % GDP')
plt.ylabel('Diabetes Prevalence')
plt.show()


- *Answer to research question:* This scatter plot visualizes the potential correlation between healthcare spending and diabetes prevalence. A visible trend could suggest a relationship between the two variables, answering our research question.

In [ ]:
# Visual 2: Box Plot of BMI by Health Expenditure Group
# Grouping expenditure into high and low categories
combined_data['Expenditure_Group'] = pd.cut(
    combined_data['Health Expenditure % GDP'], bins=[0, 5, 10, 20], labels=['Low', 'Medium', 'High']
)

# Box plot
plt.figure(figsize=(10, 6))
sns.boxplot(data=combined_data, x='Expenditure_Group', y='BMI')
plt.title('BMI Distribution by Health Expenditure Group')
plt.xlabel('Health Expenditure Group')
plt.ylabel('BMI')
plt.show()


- *Answer to research question:* This box plot shows how BMI distributions vary across different levels of health expenditure, providing further insights into the influence of healthcare spending on population health metrics.



### **5.2:** Reflection
If more time were available, I would explore additional health metrics to deepen the analysis, such as average age or income levels across countries. These factors could reveal deeper socioeconomic influences on diabetes prevalence, providing a more comprehensive view of public health drivers globally.